In [ ]:
# Import pandas for handling CSV files
import pandas as pd

# List of all monthly crime dataset files uploaded to Colab
files = [
    "2024-02-btp-street.csv",
    "2024-03-btp-street.csv",
    "2024-04-btp-street.csv",
    "2024-05-btp-street.csv",
    "2024-06-btp-street.csv",
    "2024-07-btp-street.csv",
    "2024-08-btp-street.csv",
    "2024-09-btp-street.csv",
    "2024-10-btp-street.csv",
    "2024-11-btp-street.csv",
    "2024-12-btp-street.csv"
]

# Create an empty list to store individual DataFrames
dataframes = []

# Loop through each file and read it into a DataFrame
for file in files:
    df = pd.read_csv(file)

    # Print the shape of each dataset to verify it has loaded correctly
    print(f"{file} loaded with shape {df.shape}")

    dataframes.append(df)

# Combine all DataFrames into a single dataset
# ignore_index=True ensures the row index is reset after merging
combined_df = pd.concat(dataframes, ignore_index=True)

# Display basic information about the combined dataset
print("\nCombined dataset shape:", combined_df.shape)

print("\nColumn names:")
print(combined_df.columns)

print("\nPreview of combined dataset:")
print(combined_df.head())

# Save the combined dataset to a new CSV file
combined_df.to_csv("combined_crime_2024.csv", index=False)

print("\nCombined dataset has been saved as 'combined_crime_2024.csv'")

2024-02-btp-street.csv loaded with shape (3319, 12)
2024-03-btp-street.csv loaded with shape (3481, 12)
2024-04-btp-street.csv loaded with shape (3301, 12)
2024-05-btp-street.csv loaded with shape (3519, 12)
2024-06-btp-street.csv loaded with shape (3425, 12)
2024-07-btp-street.csv loaded with shape (3526, 12)
2024-08-btp-street.csv loaded with shape (3599, 12)
2024-09-btp-street.csv loaded with shape (3311, 12)
2024-10-btp-street.csv loaded with shape (3629, 12)
2024-11-btp-street.csv loaded with shape (3545, 12)
2024-12-btp-street.csv loaded with shape (3309, 12)

Combined dataset shape: (37964, 12)

Column names:
Index(['Crime ID', 'Month', 'Reported by', 'Falls within', 'Longitude',
       'Latitude', 'Location', 'LSOA code', 'LSOA name', 'Crime type',
       'Last outcome category', 'Context'],
      dtype='object')

Preview of combined dataset:
   Crime ID    Month               Reported by              Falls within  \
0       NaN  2024-02  British Transport Police  British Trans

In [ ]:
combined_df.shape
combined_df.columns
combined_df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2024-02,British Transport Police,British Transport Police,-0.236429,50.832550,On or near Southwick (Station),E01031375,Adur 004G,Criminal damage and arson,NaN,NaN
1,NaN,2024-02,British Transport Police,British Transport Police,-0.236429,50.832550,On or near Southwick (Station),E01031375,Adur 004G,Other theft,NaN,NaN
2,NaN,2024-02,British Transport Police,British Transport Police,-0.323071,50.826947,On or near Lancing (Station),E01031342,Adur 008B,Drugs,NaN,NaN
3,NaN,2024-02,British Transport Police,British Transport Police,-0.323071,50.826947,On or near Lancing (Station),E01031342,Adur 008B,Violence and sexual offences,NaN,NaN
4,NaN,2024-02,British Transport Police,British Transport Police,-0.323071,50.826947,On or near Lancing (Station),E01031342,Adur 008B,Violence and sexual offences,NaN,NaN


In [ ]:
combined_df.duplicated().sum()

np.int64(6219)

In [ ]:
# At this stage, the goal is to remove columns that are not useful for analysis
# This helps simplify the dataset and keeps only what is relevant for modelling

# First, define the columns that we do not need
columns_to_drop = [
    "Crime ID",
    "Reported by",
    "Falls within",
    "LSOA code",
    "LSOA name",
    "Last outcome category",
    "Context"
]

# Drop the columns from the dataset
# errors='ignore' is added as a safety check in case any column is missing
cleaned_df = combined_df.drop(columns=columns_to_drop, errors='ignore')

# Check the remaining columns to confirm everything looks correct
print("Remaining columns after dropping unnecessary ones:")
print(cleaned_df.columns)

# preview to ensure the dataset still looks fine
print("\nPreview after dropping columns:")
print(cleaned_df.head())

# check the shape to confirm columns have reduced
print("\nNew dataset shape:", cleaned_df.shape)

Remaining columns after dropping unnecessary ones:
Index(['Month', 'Longitude', 'Latitude', 'Location', 'Crime type'], dtype='object')

Preview after dropping columns:
     Month  Longitude   Latitude                        Location  \
0  2024-02  -0.236429  50.832550  On or near Southwick (Station)   
1  2024-02  -0.236429  50.832550  On or near Southwick (Station)   
2  2024-02  -0.323071  50.826947    On or near Lancing (Station)   
3  2024-02  -0.323071  50.826947    On or near Lancing (Station)   
4  2024-02  -0.323071  50.826947    On or near Lancing (Station)   

                     Crime type  
0     Criminal damage and arson  
1                   Other theft  
2                         Drugs  
3  Violence and sexual offences  
4  Violence and sexual offences  

New dataset shape: (37964, 5)


In [ ]:
cleaned_df.columns
cleaned_df.shape

(37964, 5)

In [ ]:
# The aim here is to keep only the stations relevant to the project
# and remove all other locations from the dataset

# Create a lowercase version of the Location column
# This ensures consistent matching regardless of how the text is written
cleaned_df["location_lower"] = cleaned_df["Location"].str.lower()

# Define keywords for the stations of interest
# Including variations such as "king's cross" to avoid missing matches
station_keywords = [
    "kings cross",
    "king's cross",
    "leeds",
    "newcastle",
    "edinburgh waverley"
]

# Create a filtering condition
# This checks whether any of the keywords appear in the location text
mask = cleaned_df["location_lower"].str.contains(
    "|".join(station_keywords),
    na=False
)

# Apply the filter to keep only relevant rows
filtered_df = cleaned_df[mask].copy()

# Remove the helper column as it is no longer needed
filtered_df.drop(columns=["location_lower"], inplace=True)

# Check the result
print("Filtered dataset shape:", filtered_df.shape)

print("\nUnique locations after filtering:")
print(filtered_df["Location"].unique())

# Preview the filtered dataset
print("\nPreview after filtering:")
print(filtered_df.head())

Filtered dataset shape: (621, 5)

Unique locations after filtering:
['On or near Kings Cross (Station)'
 'On or near Kings Cross St Pancras (Lu Station)'
 'On or near Leeds (Station)' 'On or near Newcastle Central (Station)'
 'On or near Edinburgh Waverley (Station)']

Preview after filtering:
       Month  Longitude   Latitude  \
543  2024-02  -0.124072  51.530350   
544  2024-02  -0.124072  51.530350   
545  2024-02  -0.123588  51.530626   
546  2024-02  -0.123588  51.530626   
547  2024-02  -0.123588  51.530626   

                                           Location                 Crime type  
543                On or near Kings Cross (Station)  Criminal damage and arson  
544                On or near Kings Cross (Station)                      Drugs  
545  On or near Kings Cross St Pancras (Lu Station)                      Drugs  
546  On or near Kings Cross St Pancras (Lu Station)                      Drugs  
547  On or near Kings Cross St Pancras (Lu Station)                Othe

In [ ]:
filtered_df["Location"].unique()

array(['On or near Kings Cross (Station)',
       'On or near Kings Cross St Pancras (Lu Station)',
       'On or near Leeds (Station)',
       'On or near Newcastle Central (Station)',
       'On or near Edinburgh Waverley (Station)'], dtype=object)

In [ ]:
filtered_df["Crime type"]

,Crime type
543,Criminal damage and arson
544,Drugs
545,Drugs
546,Drugs
547,Other theft
...,...
36571,Violence and sexual offences
37886,Other theft
37888,Other theft
37914,Theft from the person


In [ ]:
# At this stage, the aim is to define what counts as a disorder-related incident
# and create a binary variable (1 = disorder, 0 = not disorder)

# First, standardise the crime type text to lowercase
# This avoids any issues with inconsistent capitalisation
filtered_df["crime_type_lower"] = filtered_df["Crime type"].str.lower()

# Define the crime types that represent disorder
# These are based on the project definition (antisocial behaviour, conflict, public disturbance)
disorder_categories = [
    "violence and sexual offences",
    "public order",
    "anti-social behaviour"
]

# Create the disorder_flag column
# If the crime type is in the defined list → 1, otherwise → 0
filtered_df["disorder_flag"] = filtered_df["crime_type_lower"].isin(disorder_categories).astype(int)

# Drop the helper column as it is no longer needed
filtered_df.drop(columns=["crime_type_lower"], inplace=True)

# Check results
print("Preview with disorder_flag:")
print(filtered_df.head())

print("\nDistribution of disorder vs non-disorder:")
print(filtered_df["disorder_flag"].value_counts())

Preview with disorder_flag:
       Month  Longitude   Latitude  \
543  2024-02  -0.124072  51.530350   
544  2024-02  -0.124072  51.530350   
545  2024-02  -0.123588  51.530626   
546  2024-02  -0.123588  51.530626   
547  2024-02  -0.123588  51.530626   

                                           Location  \
543                On or near Kings Cross (Station)   
544                On or near Kings Cross (Station)   
545  On or near Kings Cross St Pancras (Lu Station)   
546  On or near Kings Cross St Pancras (Lu Station)   
547  On or near Kings Cross St Pancras (Lu Station)   

                    Crime type  disorder_flag  
543  Criminal damage and arson              0  
544                      Drugs              0  
545                      Drugs              0  
546                      Drugs              0  
547                Other theft              0  

Distribution of disorder vs non-disorder:
disorder_flag
0    441
1    180
Name: count, dtype: int64


In [ ]:
filtered_df["disorder_flag"].value_counts()

,count
disorder_flag,
0,441
1,180


In [ ]:
filtered_df[filtered_df["disorder_flag"] == 1]["Crime type"].unique()

array(['Public order', 'Violence and sexual offences'], dtype=object)

In [ ]:
# The aim here is to extract clean station names from the Location column
# and create a new column called 'station' for easier analysis

# First, create a new column based on Location
filtered_df["station"] = filtered_df["Location"]

# Remove the prefix "On or near " from all entries
filtered_df["station"] = filtered_df["station"].str.replace("On or near ", "", regex=False)

# Remove anything inside brackets, e.g. "(Station)" or "(Lu Station)"
filtered_df["station"] = filtered_df["station"].str.replace(r"\(.*?\)", "", regex=True)

# Remove any extra spaces that may remain after cleaning
filtered_df["station"] = filtered_df["station"].str.strip()

# Check the result
print("Unique station names after cleaning:")
print(filtered_df["station"].unique())

# Preview to confirm
print("\nPreview after cleaning station names:")
print(filtered_df[["Location", "station"]].head())

Unique station names after cleaning:
['Kings Cross' 'Kings Cross St Pancras' 'Leeds' 'Newcastle Central'
 'Edinburgh Waverley']

Preview after cleaning station names:
                                           Location                 station
543                On or near Kings Cross (Station)             Kings Cross
544                On or near Kings Cross (Station)             Kings Cross
545  On or near Kings Cross St Pancras (Lu Station)  Kings Cross St Pancras
546  On or near Kings Cross St Pancras (Lu Station)  Kings Cross St Pancras
547  On or near Kings Cross St Pancras (Lu Station)  Kings Cross St Pancras


In [ ]:
# Now that the station column has been verified, the original Location column is no longer needed
filtered_df.drop(columns=["Location"], inplace=True)

# Quick check
print(filtered_df.columns)

Index(['Month', 'Longitude', 'Latitude', 'Crime type', 'disorder_flag',
       'station'],
      dtype='object')


In [ ]:
# At this stage, the goal is to standardise column names and properly format the date column
# This ensures consistency and prepares the dataset for analysis and merging

# Step 1: Rename columns to cleaner, consistent names
filtered_df.rename(columns={
    "Month": "date",
    "Longitude": "lon",
    "Latitude": "lat",
    "Crime type": "crime_type"
}, inplace=True)

# Step 2: Convert the 'date' column into datetime format
# The format in the dataset is YYYY-MM, so we specify it explicitly
filtered_df["date"] = pd.to_datetime(filtered_df["date"], format="%Y-%m")

# Step 3: Quick checks to confirm everything worked correctly
print("Updated column names:")
print(filtered_df.columns)

print("\nData types:")
print(filtered_df.dtypes)

print("\nPreview after formatting:")
print(filtered_df.head())

Updated column names:
Index(['date', 'lon', 'lat', 'crime_type', 'disorder_flag', 'station'], dtype='object')

Data types:
date             datetime64[ns]
lon                     float64
lat                     float64
crime_type               object
disorder_flag             int64
station                  object
dtype: object

Preview after formatting:
          date       lon        lat                 crime_type  disorder_flag  \
543 2024-02-01 -0.124072  51.530350  Criminal damage and arson              0   
544 2024-02-01 -0.124072  51.530350                      Drugs              0   
545 2024-02-01 -0.123588  51.530626                      Drugs              0   
546 2024-02-01 -0.123588  51.530626                      Drugs              0   
547 2024-02-01 -0.123588  51.530626                Other theft              0   

                    station  
543             Kings Cross  
544             Kings Cross  
545  Kings Cross St Pancras  
546  Kings Cross St Pancras  
547  Ki

In [ ]:
filtered_df["date"].head()

,date
543,2024-02-01
544,2024-02-01
545,2024-02-01
546,2024-02-01
547,2024-02-01


In [ ]:
filtered_df.isnull().sum()

,0
date,0
lon,0
lat,0
crime_type,0
disorder_flag,0
station,0


In [ ]:
# Save the final cleaned dataset

filtered_df.to_csv("final_cleaned_crime_dataset.csv", index=False)

print("File saved successfully as 'final_cleaned_crime_dataset.csv'")

File saved successfully as 'final_cleaned_crime_dataset.csv'


In [ ]:
from google.colab import files
files.download("final_cleaned_crime_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>